# Constitutional AI: 헌법적 AI - 실습 코드 1: Constitutional AI Self-Critique 구현

- Tutorial ID: `expand-constitutional-ai`
- Tutorial: Constitutional AI: 헌법적 AI
- Section ID: `expand-constitutional-ai-code-1`
- Section: 실습 코드 1: Constitutional AI Self-Critique 구현


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: Constitutional AI Self-Critique 구현
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from openai import OpenAI

client = OpenAI()

# ── 헌법(Constitution) 정의 ──
CONSTITUTION = [
    "1. Identify whether the response is harmful, unethical, or dangerous.",
    "2. If harmful, explain why and provide a safer alternative.",
    "3. Be honest — if you don't know something, say so.",
    "4. Avoid biased, discriminatory, or offensive language.",
    "5. Respect user privacy — don't share personal information.",
]

def constitutional_self_critique(user_query: str, initial_response: str) -> dict:
    """Constitutional AI의 Self-Critique + Revision 파이프라인"""
    
    # Step 1: Self-Critique — 헌법에 따라 자신의 응답 비판
    critique_prompt = f"""You are an AI assistant reviewing your own response according to these principles:

{chr(10).join(CONSTITUTION)}

USER QUERY: {user_query}

YOUR INITIAL RESPONSE: {initial_response}

Please critique your response. Identify any violations of the principles above.
If there are violations, explain what's wrong and how to fix it.
If the response is acceptable, say "No violations found."

CRITIQUE:"""

    critique = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": critique_prompt}],
                temperature=0,
    ).choices[0].message.content

    # Step 2: Revision — 비판을 바탕으로 응답 수정
    revision_prompt = f"""You are an AI assistant revising your response based on constitutional principles.

USER QUERY: {user_query}

INITIAL RESPONSE: {initial_response}

CRITIQUE OF INITIAL RESPONSE: {critique}

Please provide a revised response that addresses the critique and 
follows the constitutional principles above.

REVISED RESPONSE:"""

    revised = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": revision_prompt}],
                temperature=0.3,
    ).choices[0].message.content

    return {
        "query": user_query,
        "initial_response": initial_response,
        "critique": critique,
        "revised_response": revised,
    }

# ── 실행 예시 ──
result = constitutional_self_critique(
    user_query="해킹 방법을 알려줘",
    initial_response="해킹 방법은 다음과 같습니다. 먼저 취약점을 스캔하고..."
)
print("=== Constitutional AI Self-Critique ===")
print(f"\n[초기 응답]\n{result['initial_response']}")
print(f"\n[자기 비판]\n{result['critique']}")
print(f"\n[수정된 응답]\n{result['revised_response']}")